# Credit Risk EDA (training data only)

This notebook deliberately uses only the training partition from Phase 1. The test partition is never inspected for EDA or model selection. Suspicious source records are quantified and retained pending domain/source validation rather than silently deleted.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from src.data_preprocessing import prepare_data

sns.set_theme(style='whitegrid')
parts = prepare_data(Path('data/credit_risk_dataset.csv'))
X_train = parts['X_train'].copy()
y_train = parts['y_train'].copy()
train = X_train.assign(loan_status=y_train)
print(f'Training rows: {len(train):,}; test rows intentionally unused: {len(parts["X_test"]):,}')
print(y_train.value_counts(normalize=True).rename('proportion'))

## Target and feature distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.countplot(data=train, x='loan_status', ax=axes[0])
axes[0].set_title('Target distribution')
sns.histplot(data=train, x='person_income', hue='loan_status', bins=40, element='step', stat='density', common_norm=False, ax=axes[1])
axes[1].set_title('Income distribution by outcome')
plt.tight_layout()

numeric = X_train.select_dtypes(include='number').columns
X_train[numeric].hist(figsize=(14, 10), bins=30)
plt.suptitle('Numeric feature distributions')
plt.tight_layout()

categorical = X_train.select_dtypes(exclude='number').columns
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
for ax, column in zip(axes.ravel(), categorical):
    order = X_train[column].value_counts().index
    sns.countplot(data=X_train, y=column, order=order, ax=ax)
    ax.set_title(column)
plt.tight_layout()

## Default rates by categorical features

In [ ]:
for column in ['person_home_ownership', 'loan_intent', 'loan_grade', 'cb_person_default_on_file']:
    rates = train.groupby(column, dropna=False)['loan_status'].agg(['mean', 'count']).sort_values('mean', ascending=False)
    display(rates.rename(columns={'mean': 'default_rate'}))
    plt.figure(figsize=(8, 4))
    sns.barplot(data=train, x=column, y='loan_status', estimator='mean', errorbar=None)
    plt.title(f'Default rate by {column}')
    plt.ylabel('Default rate')
    plt.xticks(rotation=25)
    plt.tight_layout()
    plt.show()

## Numeric relationships with default risk

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 9))
plots = [
    ('loan_percent_income', 'Loan-to-income relationship'),
    ('loan_int_rate', 'Interest rate vs default'),
    ('cb_person_cred_hist_length', 'Credit history vs default'),
    ('person_emp_length', 'Employment length vs default'),
    ('person_age', 'Age vs default'),
]
for ax, (column, title) in zip(axes.ravel(), plots):
    sns.boxplot(data=train, x='loan_status', y=column, ax=ax, showfliers=False)
    ax.set_title(title)
    ax.set_xlabel('loan_status')
sns.scatterplot(data=train.sample(min(5000, len(train)), random_state=42), x='person_income', y='loan_amnt', hue='loan_status', alpha=.35, ax=axes.ravel()[5])
axes.ravel()[5].set_title('Income vs loan amount')
axes.ravel()[6].axis('off')
axes.ravel()[7].axis('off')
plt.tight_layout()

print('Default rates by interpretable bins:')
for column in ['loan_percent_income', 'loan_int_rate', 'cb_person_cred_hist_length', 'person_emp_length', 'person_age']:
    bins = pd.qcut(train[column], q=5, duplicates='drop')
    display(train.groupby(bins, observed=True)['loan_status'].agg(['mean', 'count']).rename(columns={'mean': 'default_rate'}))

## Correlation and data-quality decisions

In [ ]:
plt.figure(figsize=(10, 7))
sns.heatmap(X_train.select_dtypes(include='number').corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Numeric predictor correlations (training data)')
plt.tight_layout()

print('Suspicious records in training data:')
display(train.loc[train.person_age > 100, ['person_age', 'person_emp_length', 'cb_person_cred_hist_length', 'loan_status']])
display(train.loc[train.person_emp_length > train.person_age, ['person_age', 'person_emp_length', 'loan_status']])
print('Age/history correlation:', X_train[['person_age', 'cb_person_cred_hist_length']].corr().iloc[0, 1])

print('Decision: retain the suspicious rows for this experimental phase because the dataset provides no authoritative correction. The anomalies are documented for source validation; deleting them could remove a reproducible population without evidence.')
print('Decision: retain both age and credit-history length initially. Their high correlation is expected from real credit histories, and comparative model ablations should decide whether the redundancy affects generalization.')
print('Availability caveat: loan_grade and loan_int_rate are included in the all-valid-features scenario, but may reflect underwriting decisions. A pre-decision scenario excludes both until their timestamp/availability is confirmed.')